In [1]:
import time
import numpy as np
import pandas as pd
import yfinance as yf
from pytrends.request import TrendReq

In [2]:
tickers = ["SPY",'CL=F','^VIX','MRNA','BNTX']
stock_trends = yf.download(tickers, start="2015-01-01")

stock_trends.columns = ["_".join(col) for col in stock_trends.columns]

# features
for ticker in tickers:
    stock_trends[ticker+'_log_ret'] = np.log( stock_trends['Close_'+ticker] / stock_trends['Close_'+ticker].shift(1) )
    stock_trends[ticker+'_log_ret_t+1'] = stock_trends[ticker+'_log_ret'].shift(-1)
    stock_trends[ticker+'_vol_10'] = stock_trends[ticker+'_log_ret'].rolling(window=10).std()
    stock_trends[ticker+'_vol_10_t+1'] = stock_trends[ticker+'_vol_10'].shift(-1)
    stock_trends[ticker+'_shock_10'] = stock_trends[ticker+'_log_ret'] / stock_trends[ticker+'_vol_10']
    stock_trends[ticker+'_shock_10_t+1'] = stock_trends[ticker+'_shock_10'].shift(-1)
    stock_trends = stock_trends.drop(columns=[i+'_'+ticker for i in ['Open','Close','High','Low','Volume']])

stock_trends = stock_trends.dropna()
stock_trends.to_csv('stock_trends.csv')
stock_trends

[*********************100%***********************]  5 of 5 completed
c:\Users\athar\Desktop\gtsf project\venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,SPY_log_ret,SPY_log_ret_t+1,SPY_vol_10,SPY_vol_10_t+1,SPY_shock_10,SPY_shock_10_t+1,CL=F_log_ret,CL=F_log_ret_t+1,CL=F_vol_10,CL=F_vol_10_t+1,...,MRNA_vol_10,MRNA_vol_10_t+1,MRNA_shock_10,MRNA_shock_10_t+1,BNTX_log_ret,BNTX_log_ret_t+1,BNTX_vol_10,BNTX_vol_10_t+1,BNTX_shock_10,BNTX_shock_10_t+1
Date,,,,,,,,,,,,,,,,,,,,,
2019-10-24,0.001632,0.004087,0.005217,0.004488,0.312902,0.910577,0.004635,0.007618,0.016686,0.015730,...,0.026595,0.025391,-0.696128,0.984652,0.066344,-0.070320,0.111569,0.114591,0.594645,-0.613662
2019-10-25,0.004087,0.005621,0.004488,0.004511,0.910577,1.246036,0.007618,-0.015115,0.015730,0.014885,...,0.025391,0.024331,0.984652,0.048297,-0.070320,-0.065260,0.114591,0.113130,-0.613662,-0.576858
2019-10-28,0.005621,-0.000297,0.004511,0.003736,1.246036,-0.079371,-0.015115,-0.004850,0.014885,0.013798,...,0.024331,0.025100,0.048297,-0.093689,-0.065260,0.032280,0.113130,0.113060,-0.576858,0.285514
2019-10-29,-0.000297,0.003062,0.003736,0.003603,-0.079371,0.849893,-0.004850,-0.008680,0.013798,0.014287,...,0.025100,0.023123,-0.093689,0.380146,0.032280,-0.012430,0.113060,0.113501,0.285514,-0.109512
2019-10-30,0.003062,-0.002667,0.003603,0.003852,0.849893,-0.692369,-0.008680,-0.016112,0.014287,0.015202,...,0.023123,0.026405,0.380146,-0.871661,-0.012430,0.001785,0.113501,0.113564,-0.109512,0.015719
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-13,0.009725,0.012111,0.010807,0.010104,0.899917,1.198648,0.025659,-0.081996,0.072590,0.075789,...,0.026973,0.027420,-0.204263,1.522154,0.012487,0.017529,0.019309,0.018381,0.646695,0.953660
2026-04-14,0.012111,0.007860,0.010104,0.007517,1.198648,1.045692,-0.081996,0.000110,0.075789,0.075874,...,0.027420,0.023976,1.522154,1.106034,0.017529,0.010714,0.018381,0.015671,0.953660,0.683650
2026-04-15,0.007860,0.002454,0.007517,0.007673,1.045692,0.319860,0.000110,0.036567,0.075874,0.077302,...,0.023976,0.022716,1.106034,0.339436,0.010714,0.026293,0.015671,0.016222,0.683650,1.620796


In [ ]:
pytrends = TrendReq()

kws = ['oil price', 'war', 'stock market', 'Iran', 'Hormuz']

trends = pd.DataFrame()
first = False
time.sleep(60)
for i in kws:
    last_request = time.time()
    pytrends.build_payload(
        kw_list = [i],
        timeframe='today 5-y'
    )

    attention = pytrends.interest_over_time()
    if first:
        trends = attention
    else:
        trends[i]=attention[i]

    print('fetched '+i)   
    until_next_request = last_request+0-time.time()
    if until_next_request>0 and i!='Hormuz':
        time.sleep(until_next_request)


fetched oil price
fetched war
fetched stock market
fetched Iran
fetched Hormuz


In [ ]:
trends

In [ ]:
trends = trends.resample("D").ffill()
for col in trends.columns:
    trends[col]=trends[col].rolling(window=7).mean()
trends['Attention_mean'] = trends[kws].mean(axis=1)
for i in kws:
    trends[i+'_diff'] = trends[i].diff()
trends.dropna()
trends.to_csv('trends.csv', index=True)
trends

,oil price,war,stock market,Iran,Hormuz,Attention_mean,oil price_diff,war_diff,stock market_diff,Iran_diff,Hormuz_diff
date,,,,,,,,,,,
2021-04-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-04-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-04-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-04-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-04-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,41.857143,45.285714,24.428571,26.714286,86.285714,44.914286,-1.285714,-1.428571,-0.142857,-2.571429,-0.428571
2026-04-16,40.571429,43.857143,24.285714,24.142857,85.857143,43.742857,-1.285714,-1.428571,-0.142857,-2.571429,-0.428571
2026-04-17,39.285714,42.428571,24.142857,21.571429,85.428571,42.571429,-1.285714,-1.428571,-0.142857,-2.571429,-0.428571


# Chocolate Trends Data

In [3]:
from pytrends.request import TrendReq
import pandas as pd
import numpy as np
import time

pytrends = TrendReq()

kws = ['FDA approval', 'clinical trial', 'phase 2',
       'phase 3', 'trial results', 'efficacy',
       'biotech','Moderna','Pfizer','BioNTech',
       'drug approval', 'new drug']

start_date = pd.to_datetime("2021-01-01")
end_date   = pd.to_datetime("2026-04-17")

window_months = 6
overlap_months = 2   # IMPORTANT

final_df = None

current_start = start_date

while current_start < end_date:

    current_end = current_start + pd.DateOffset(months=window_months)
    if current_end > end_date:
        current_end = end_date

    timeframe = f'{current_start.date()} {current_end.date()}'
    print(f"\nFetching window: {timeframe}")

    window_df = pd.DataFrame()

    time.sleep(60)  # avoid rate limit

    for kw in kws:
        print(f"Fetching {kw}")

        pytrends.build_payload(
            kw_list=[kw],
            timeframe=timeframe
        )

        data = pytrends.interest_over_time()
        data = data.drop(columns=['isPartial'], errors='ignore')

        if not data.empty:
            window_df[kw] = data[kw]
        else:
            window_df[kw] = np.nan

    # --- STITCHING LOGIC ---
    if final_df is None:
        final_df = window_df.copy()
    else:
        # find overlap
        overlap_idx = final_df.index.intersection(window_df.index)

        if len(overlap_idx) > 0:
            # compute scaling per column
            scale_factors = {}

            for kw in kws:
                old_vals = final_df.loc[overlap_idx, kw]
                new_vals = window_df.loc[overlap_idx, kw]

                # avoid divide by zero / NaN
                if new_vals.mean() > 0:
                    scale = old_vals.mean() / new_vals.mean()
                else:
                    scale = 1.0

                scale_factors[kw] = scale

            # rescale new window
            for kw in kws:
                window_df[kw] = window_df[kw] * scale_factors[kw]

        # append only non-overlapping part
        window_df = window_df.loc[~window_df.index.isin(final_df.index)]
        final_df = pd.concat([final_df, window_df])

    # move forward WITH overlap
    current_start = current_end - pd.DateOffset(months=overlap_months)

# remove duplicates (safety)
final_df = final_df[~final_df.index.duplicated(keep='first')]

# --- FEATURE ENGINEERING ---
for kw in kws:
    final_df[kw + '_diff'] = final_df[kw].diff()

    rolling_mean = final_df[kw].rolling(window=10).mean()
    rolling_std = final_df[kw].rolling(window=10).std()

    final_df[kw + '_shock_10'] = (final_df[kw] - rolling_mean) / rolling_std

# save
final_df.to_csv('trends_stitched.csv')

final_df


Fetching window: 2021-01-01 2021-07-01
Fetching FDA approval
Fetching clinical trial
Fetching phase 2
Fetching phase 3
Fetching trial results
Fetching efficacy
Fetching biotech
Fetching Moderna
Fetching Pfizer
Fetching BioNTech
Fetching drug approval
Fetching new drug

Fetching window: 2021-05-01 2021-11-01
Fetching FDA approval
Fetching clinical trial
Fetching phase 2
Fetching phase 3
Fetching trial results
Fetching efficacy
Fetching biotech
Fetching Moderna
Fetching Pfizer
Fetching BioNTech
Fetching drug approval
Fetching new drug

Fetching window: 2021-09-01 2022-03-01
Fetching FDA approval
Fetching clinical trial
Fetching phase 2
Fetching phase 3
Fetching trial results
Fetching efficacy
Fetching biotech
Fetching Moderna
Fetching Pfizer
Fetching BioNTech
Fetching drug approval
Fetching new drug

Fetching window: 2022-01-01 2022-07-01
Fetching FDA approval
Fetching clinical trial
Fetching phase 2
Fetching phase 3
Fetching trial results
Fetching efficacy
Fetching biotech
Fetching Mod

KeyboardInterrupt: 